In [1]:
import numpy as np
from scipy.io.netcdf import netcdf_file
from scipy.interpolate import griddata
from datetime import datetime
from os.path import join
from MyLib import GPSConverter, es_calc
from pandas import to_datetime

C:\Users\irani\AppData\Local\Temp\ipykernel_17916\3469418119.py:2: DeprecationWarning: Please use `netcdf_file` from the `scipy.io` namespace, the `scipy.io.netcdf` namespace is deprecated.
  from scipy.io.netcdf import netcdf_file


In [2]:
# configuration
#name = ("atemp",  "aqh", "swdown",     "lwdown", "apressure")
#ncvar = ("T_2M",  "AQH",   "GLOB",   "LW_IN_TG",        "PS")
#shift = (    0.,     0.,       0.,           0.,          0.)
#factor = (   1.,     1.,        1,           1.,          1.)
#name = ("atemp",  "aqh", "swdown",     "lwdown",   "precip", "apressure")
#ncvar = ("T_2M",  "AQH",   "GLOB",   "LW_IN_TG", "TOT_PREC",        "PS")
#shift = (    0.,     0.,       0.,           0.,         0.,          0.)
#factor = (   1.,     1.,        1,           1.,   1./3.6e6,          1.)
name = ("atemp",  "aqh", "swdown",     "lwdown", "apressure")
ncvar = ("T_2M",  "AQH",   "GLOB",   "LW_IN_TG",        "PS")
shift = (    0.,     0.,       0.,           0.,          0.)
factor = (   1.,     1.,        1,           1.,          1.)
vrange = ([230, 320], [1e-5, 0.1], [0, 2e3], [0, 2e3], [5e4, 2e5])

# mitgcm_grid_file = "E:/ALPLakes/Codes/MITgcm_input_output/input_generator/LemanGrid_HR50.nc"
# mitgcm_grid_file = "G:/ALPLakes/D3D/lac_de_joux/LDJ_10days_2020/mitgcm_format/joux_grid.nc"
mitgcm_grid_file = "C:/Users/irani/Documents/PT_D3DtoMITgcm/Lake_StMoritz/10days/MITgcm_format/StMoritzGrid.nc"
out_fmt = "float"

shf = dict(zip(name, shift))
fac = dict(zip(name, factor))
var = dict(zip(name, ncvar))
vrn = dict(zip(ncvar, vrange))

# extract COSMO grid info
def get_MITgcm_grid(mitgcm_file):
    #%Extract geographical data from the first COSMO-2 file
    #date = datevec(dateini); %Date vector
    #FileName = [datapath sprintf('%i',date(1)) '\cosmo2_epfl_lakes_' sprintf('%i%02i%02i',date(1:3)) '.nc'];
    f = netcdf_file(mitgcm_file, 'r')
    # Must transpose arrays to be consistent with Fortran ordering
    xC = f.variables["XC"][:].copy().T
    yC = f.variables["YC"][:].copy().T
    f.close()
    return xC, yC

# extract COSMO grid info
def get_COSMO_grid(cosmo_file):
    #%Extract geographical data from the first COSMO-2 file
    #date = datevec(dateini); %Date vector
    #FileName = [datapath sprintf('%i',date(1)) '\cosmo2_epfl_lakes_' sprintf('%i%02i%02i',date(1:3)) '.nc'];
    f = netcdf_file(cosmo_file, 'r')
    lon = f.variables["lon_1"][:].copy()
    lat = f.variables["lat_1"][:].copy()
    f.close()
    converter = GPSConverter()
    x = np.zeros_like(lon)
    y = np.zeros_like(lat)
    for jj in range(x.shape[1]):
        for ii in range(x.shape[0]):
            x[ii,jj], y[ii,jj], _ = converter.WGS84toLV03(
                                    lat[ii, jj], lon[ii,jj], 0.0)
    return x, y

# get time axis info from COSMO file
def get_time(ncf):
    time = ncf.variables["time"][:].copy().astype("int")
    refdate = ncf.variables["time"].units.decode('utf8')
    refdate = refdate.split()
    if refdate[0] == "seconds":
        units = "[s]"
    elif refdate[0] == "hours":
        units = "[h]"
    else:
        raise ValueError("Only seconds are implemented")
    refdate = refdate[2].rstrip(",; ") + " " + refdate[3].rstrip(",; ")
    return np.array([np.datetime64(refdate) + np.timedelta64(t, units)
                    for t in time])

def check_var(v, varrange):
    if (v.min() < varrange[0]) | (v.max() > varrange[1]):
        return True
    else:
        return False

def cosmo2delft3d(data_path,data_root, start, end, xinfo, yinfo,output_path): # revised by Fazel
    

    # Time range to be extracted
    date_start = np.datetime64(start)
    date_end = np.datetime64(end)

    # Get grid info
    fname = data_root + to_datetime(date_start).strftime("%Y%m%d") + ".nc"
    fname = join(data_path,fname)

    # generate output grid
    xM, yM = get_MITgcm_grid(mitgcm_grid_file)

    # open the output files
    fls = {}
    for e in name:
        out_name = output_path+"%s_%s_%s.bin" % \
                   (e, start.replace('-',''), end.replace('-',''))
        fls[e] = open(out_name, "ab")

    # loop over COSMO-2 data
    today = date_start
    missing_day = False
    while today<=date_end:
#         # This is to follow a change in the naming convention
#         # so stupid one cannot believe
#         if today <= np.datetime64("2015-05-10"):
#             date_file = today
#         else:
#             date_file = today + np.timedelta64(1, 'D')
#       
        date_file = today  
        fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
        fname = join(data_path, fname)
        try:
            xc, yc = get_COSMO_grid(fname)
            ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
                           (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
            in_f = netcdf_file(fname, 'r')
        except IOError:
            print("!!! Missing entire day of forcing !!!\nRepeating the previous day.")
            date_file -= np.timedelta64(1, 'D')
            fname = data_root + to_datetime(date_file).strftime("%Y%m%d") + ".nc"
            fname = join(data_path, fname)
            xc, yc = get_COSMO_grid(fname)
            ind = np.where((xc>=xinfo[0]-15e3) & (xc<=xinfo[1]+15e3) & 
                           (yc>=yinfo[0]-15e3) & (yc<=yinfo[1]+15e3))
            in_f = netcdf_file(fname, 'r')
            missing_day = True
        print("\ndate: %s\nfile: %s" % (today, fname))

        nc_time = get_time(in_f)
#         print(nc_time)
        if missing_day: #modified by Fazel
            req_time = np.arange(today - np.timedelta64(1, 'D') ,
                                 today - np.timedelta64(1, 'D') + np.timedelta64(1, 'D'),
                                 np.timedelta64(1, 'h'))
            missing_day = False
        else:
            # modified by Fazel
            req_time = np.arange(today,
                                 today + np.timedelta64(1, 'D'),
                                 np.timedelta64(1, 'h'))
    
        # write output files
        for e, f in fls.items():
            v = var[e]
            try:
                if len(in_f.variables[v].shape) == 4:
                    data = in_f.variables[v][:24, 0, ind[0], ind[1]]
                    data *= fac[e]
                    data += shf[e]
                elif len(in_f.variables[v].shape) == 3:
                    data = in_f.variables[v][:, ind[0], ind[1]]
                    data *= fac[e]
                    data += shf[e]
                if v == "TOT_PREC":
                    data[data < 0] = 0.0
                if v == "PS":
                    # we check if some idiot suddenly decided
                    # to change the units of pressure
                    if np.max(data) < 1e4:
                        data *= 100
                if v == "GLOB":
                    # we check if some idiot suddenly decided
                    # to change the units of pressure
                    if np.min(data) < 0:
                        data *= 0
                if v == "LW_IN_TG":
                    # we check if some idiot suddenly decided
                    # to change the units of pressure
                    if np.min(data) < 0:
                        data = data*0+300 # must be improved
            except KeyError:
                if v == "AQH":
                    if len(in_f.variables["T_2M"].shape) == 3:
                        relh = in_f.variables["RELHUM_2M"][:, ind[0], ind[1]]
                        tmp = in_f.variables["T_2M"][:, ind[0], ind[1]] - 273.15
                        ps = in_f.variables["PS"][:, ind[0], ind[1]]
                        if np.max(ps) < 1e4:
                            ps *= 100
                        ew = es_calc(tmp) * 0.01 * relh
                        rv = 0.622 * ew / (ps - ew)
                        data = rv / (1 + rv)
                    elif len(in_f.variables["T_2M"].shape) == 4:
                        relh = in_f.variables["RELHUM_2M"][:24,0, ind[0], ind[1]]
                        tmp = in_f.variables["T_2M"][:24,0, ind[0], ind[1]] - 273.15
                        ps = in_f.variables["PS"][:24,0, ind[0], ind[1]]
                        if np.max(ps) < 1e4:
                            ps *= 100
                        ew = es_calc(tmp) * 0.01 * relh
                        rv = 0.622 * ew / (ps - ew)
                        data = rv / (1 + rv)
                else:
                    raise KeyError("'%s' variable not found" % v)
            if check_var(data, vrn[v]):
                print("%s range: %.3g - %.3g" % (v, data.min(), data.max()))
            n_miss = 0
            for hh in req_time:
                try:
                    index = np.where(nc_time == hh)[0][0]
                    out_data = griddata((xc[ind[0], ind[1]], yc[ind[0], ind[1]]),
                                        data[index, ...],
                                        (xM.ravel(), yM.ravel()),
                                        fill_value=np.nan, method="linear")
                    bad = np.isnan(out_data)
                    # if there are missing points, we fill them with
                    # nearest neighbour values
                    if np.any(bad > 0):
                        out_data[bad] = griddata((xc[ind[0], ind[1]],
                                                  yc[ind[0], ind[1]]),
                                                  data[index, ...],
                                                  (xM.ravel()[bad],
                                                   yM.ravel()[bad]),
                                                  method="nearest")
                    out_data = np.reshape(out_data, xM.shape)
                except IndexError:
                    print("Warning: missing data in file!")
                    # we do not need to do anything, we will just be writing the
                    # last out_data array we computed
                    n_miss += 1
                # write to file
                for kk in range(xM.shape[1]):
                    out_data[:,kk].tofile(f)
                    print(out_data.mean())
            if n_miss > 0:
                print("Not all expected times are available in this file:\n %s"
                      % nc_time)
                if n_miss == 24:
                    raise ValueError("Nothing in this file")
        
        in_f.close()

        today += np.timedelta64(1, 'D')

    # close files
    for f in fls.values():
        f.close()

In [3]:
# cosmo2delft3d('G:/ALPLakes/COSMO/2021/','cosmo2_epfl_lakes_','2021-07-26','2021-09-10',[500000.0,563000.0,1000.0],[116500.0,138700.0,1000.0],'G:/ALPLakes/MITgcm/grid50_64cores_secchi_updated/binary_data/')
# cosmo2delft3d('F:/COSMO/2021/','cosmo2_epfl_lakes_','2021-09-02','2021-09-10',[500000.0,563000.0,1000.0],[116500.0,138700.0,1000.0],'F:/Datalakes50_Fazel/Hydrodynamics/binary_data/')
# cosmo2delft3d('H:/COSMO/2022/','cosmo2_epfl_lakes_','2022-08-01','2022-10-04',[500000.0,563000.0,1000.0],[116500.0,138700.0,1000.0],'F:/MITgcm_Dave/binary_data/')
cosmo2delft3d('F:/COSMO/2020/','cosmo2_epfl_lakes_','2020-054-15','2020-05-24',[506000.0,517000.0,500],[161000.0,171000.0,500],'G:/ALPLakes/D3D/lac_de_joux/LDJ_10days_2020/mitgcm_format/')


date: 2020-04-15
file: F:/COSMO/2020/cosmo2_epfl_lakes_20200415.nc
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683749
274.0294606683

272.7570812176357
272.7570812176357
272.7570812176357
272.7570812176357
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.4895523614658
272.489552

278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.4791032523746
278.479103

287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.6315825058832
287.631582

288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873286993
288.23819873

...

316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.1465398346657
316.146539

277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.7845594909536
277.784559

89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.06940583972
89800.0694

89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.94622520093
89764.9462

89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.84200447203
89484.8420